# Mintx: fine-tuning a PII token-classification model

This notebook fine-tunes a pretrained transformer to find **personal data** in web-page text, following the
Mintx labelling policy (13 labels, with public entities such as *Google* or *India* left visible). It then:

1. evaluates on three test sets (held-out templates, ai4privacy, hand-checked realistic text), against an off-the-shelf NER baseline,
2. exports the model to **ONNX**, quantizes it to **int8** and checks the accuracy cost,
3. pushes a browser-ready model (transformers.js layout) to the **Hugging Face Hub**.

**Run on Kaggle:** *Settings → Accelerator → GPU T4 ×2 (or P100)*, *Internet on*, add a secret `HF_TOKEN` (Add-ons → Secrets), then *Run all*.
**Run on Colab:** *Runtime → Change runtime type → T4 GPU*, add `HF_TOKEN` under 🔑 Secrets, then *Run all*.

Training takes about 15–25 minutes on a T4. Edit only the configuration cell below.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────────────────
BASE_MODEL    = "distilbert-base-cased"        # alternatives: "dslim/bert-base-NER" (transfer from CoNLL NER),
                                               #               "microsoft/deberta-v3-small"
RUN_NAME      = "mintx-pii-distilbert"
HF_DATASET    = "YOUR_HF_USERNAME/mintx-pii"   # uploaded with ml/push_dataset.py
HF_MODEL_REPO = "YOUR_HF_USERNAME/mintx-pii-distilbert"
PUSH_TO_HUB   = True

EPOCHS, LR, BATCH, MAX_LEN = 3, 5e-5, 32, 256
WARMUP, WEIGHT_DECAY, SEED = 0.1, 0.01, 42
RARE_LABELS     = ["BANK_ACCOUNT", "UPI_ID", "IP_ADDRESS", "SECRET", "ORGANIZATION"]
RARE_OVERSAMPLE = 4        # repeat training examples that contain a rare label
KEEP_OVERSAMPLE = 2        # repeat examples that contain public "keep" entities (explicit negatives)
USE_CLASS_WEIGHTS = False  # alternative to oversampling: inverse-sqrt-frequency weighted loss
COMPARE_BASELINE  = True   # also evaluate the off-the-shelf CoNLL NER model (dslim/bert-base-NER)
SMOKE_TEST        = False  # True: tiny subset, 1 epoch — checks the whole pipeline on CPU in minutes

import os
if os.environ.get("MINTX_SMOKE") == "1":      # used by the local pipeline check
    SMOKE_TEST, PUSH_TO_HUB = True, False

In [ ]:
# ── Environment ───────────────────────────────────────────────────────────────────────────
import sys
IN_KAGGLE = os.path.exists("/kaggle/working")
IN_COLAB = "google.colab" in sys.modules
if IN_KAGGLE or IN_COLAB:
    %pip install -q "transformers==4.57.6" "accelerate>=1.0" "seqeval==1.2.2" "onnx==1.18.0" "onnxruntime>=1.20" hf_xet

In [ ]:
import json, math, random, time, inspect, collections, shutil
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from transformers import (AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification,
                          Trainer, TrainingArguments, pipeline, set_seed)
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORK = Path("/kaggle/working") if IN_KAGGLE else Path("/content") if IN_COLAB else Path("outputs")
OUT = WORK / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
print(f"device={DEVICE}  torch={torch.__version__}  output={OUT}")
if DEVICE == "cpu" and not SMOKE_TEST:
    print("⚠️  No GPU found. Enable one (Kaggle: Settings → Accelerator) or training will take hours.")


def hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None

## 1 · Data

In [ ]:
FILES = ["train", "val", "test_synthetic", "test_ai4privacy", "test_realistic"]

def find_local():
    roots = [Path("/kaggle/input"), Path("data/processed"), Path("ml/data/processed"), Path("/content/data")]
    for root in roots:
        if root.exists():
            hit = next(root.rglob("train.jsonl"), None)
            if hit and all((hit.parent / f"{f}.jsonl").exists() for f in FILES):
                return hit.parent
    return None

data_dir = find_local()
if data_dir is None:
    from huggingface_hub import hf_hub_download
    data_dir = Path(hf_hub_download(HF_DATASET, "train.jsonl", repo_type="dataset", token=hf_token())).parent
    for f in FILES[1:] + ["label_map"]:
        hf_hub_download(HF_DATASET, f"{f}.json" + ("l" if f != "label_map" else ""), repo_type="dataset", token=hf_token())
print("data:", data_dir)

def read(name):
    with open(data_dir / f"{name}.jsonl", encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

data = {f: read(f) for f in FILES}
LABELS = json.load(open(data_dir / "label_map.json"))["labels"]
if SMOKE_TEST:
    rng = random.Random(SEED)
    data = {k: rng.sample(v, min(len(v), 300 if k == "train" else 60)) for k, v in data.items()}
    EPOCHS, MAX_LEN = 1, 128
pd.DataFrame({k: {"examples": len(v), "pii spans": sum(len(r["spans"]) for r in v),
                  "keep spans": sum(len(r["keep"]) for r in v)} for k, v in data.items()}).T

In [ ]:
id2label = {0: "O"}
for lab in LABELS:
    id2label[len(id2label)] = f"B-{lab}"
    id2label[len(id2label)] = f"I-{lab}"
label2id = {v: k for k, v in id2label.items()}
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
assert tokenizer.is_fast, "a fast tokenizer is required for character offsets"

def encode(rec):
    # Tokenize and give every token a BIO label from the character spans (keep spans stay O).
    enc = tokenizer(rec["text"], truncation=True, max_length=MAX_LEN, return_offsets_mapping=True)
    owner = np.full(len(rec["text"]), -1)
    for i, s in enumerate(rec["spans"]):
        owner[s["start"]:s["end"]] = i
    labels, prev = [], -1
    for (s, e), special in zip(enc["offset_mapping"], enc.word_ids()):
        if special is None or e <= s:
            labels.append(-100); continue
        chars = owner[s:e]
        idx = next((int(c) for c in chars if c >= 0), -1)
        if idx < 0:
            labels.append(label2id["O"]); prev = -1; continue
        tag = "I" if idx == prev else "B"
        labels.append(label2id[f"{tag}-{rec['spans'][idx]['label']}"]); prev = idx
    enc.pop("offset_mapping")
    enc["labels"] = labels
    return dict(enc)

def oversample(records):
    out = []
    for r in records:
        labels = {s["label"] for s in r["spans"]}
        times = RARE_OVERSAMPLE if labels & set(RARE_LABELS) else KEEP_OVERSAMPLE if r["keep"] else 1
        out += [r] * times
    return out

train_records = oversample(data["train"])
random.Random(SEED).shuffle(train_records)
train_ds = [encode(r) for r in train_records]
val_ds = [encode(r) for r in data["val"]]
print(f"{len(data['train']):,} training examples → {len(train_ds):,} after oversampling rare labels")

ex = data["test_realistic"][3]
enc = tokenizer(ex["text"], truncation=True, max_length=MAX_LEN)
print(list(zip(tokenizer.convert_ids_to_tokens(enc["input_ids"]), [id2label.get(i, "-") for i in encode(ex)["labels"]]))[:24])

## 2 · Fine-tuning

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL, num_labels=len(id2label), id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True)

class_weights = None
if USE_CLASS_WEIGHTS:
    counts = collections.Counter(l for ex in train_ds for l in ex["labels"] if l != -100)
    freq = np.array([counts.get(i, 1) for i in range(len(id2label))], dtype=float)
    w = 1 / np.sqrt(freq); class_weights = torch.tensor(w / w.mean(), dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        if class_weights is None:
            return super().compute_loss(model, inputs, return_outputs=return_outputs, **kwargs)
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(outputs.logits.view(-1, len(id2label)), labels.view(-1),
                                                 weight=class_weights.to(outputs.logits.device), ignore_index=-100)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    true_seq, pred_seq = [], []
    for pr, la in zip(preds, p.label_ids):
        keep = la != -100
        true_seq.append([id2label[int(x)] for x in la[keep]]); pred_seq.append([id2label[int(x)] for x in pr[keep]])
    return {"precision": precision_score(true_seq, pred_seq), "recall": recall_score(true_seq, pred_seq),
            "f1": f1_score(true_seq, pred_seq)}

args = TrainingArguments(
    output_dir=str(OUT / "checkpoints"), num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH * 2, warmup_ratio=WARMUP,
    weight_decay=WEIGHT_DECAY, eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="f1", save_total_limit=1, logging_steps=10 if SMOKE_TEST else 50,
    fp16=torch.cuda.is_available(), report_to="none", seed=SEED, dataloader_num_workers=0 if (SMOKE_TEST or os.name == "nt") else 2)
trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                          data_collator=DataCollatorForTokenClassification(tokenizer), processing_class=tokenizer,
                          compute_metrics=compute_metrics)
t0 = time.time()
trainer.train()
train_minutes = (time.time() - t0) / 60
trainer.save_model(str(OUT / "model")); tokenizer.save_pretrained(str(OUT / "model"))
print(f"trained in {train_minutes:.1f} min")

In [ ]:
log = pd.DataFrame(trainer.state.log_history)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
tr = log.dropna(subset=["loss"]) if "loss" in log else log.iloc[0:0]
ev = log.dropna(subset=["eval_loss"])
ax[0].plot(tr["epoch"], tr["loss"], label="train loss"); ax[0].plot(ev["epoch"], ev["eval_loss"], "o-", label="validation loss")
ax[0].set_xlabel("epoch"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(ev["epoch"], ev["eval_f1"], "o-", label="F1"); ax[1].plot(ev["epoch"], ev["eval_precision"], "o--", label="precision")
ax[1].plot(ev["epoch"], ev["eval_recall"], "o--", label="recall"); ax[1].set_ylim(0, 1); ax[1].set_xlabel("epoch")
ax[1].set_title("Validation (entity-level, seqeval)"); ax[1].legend()
plt.tight_layout(); plt.savefig(OUT / "training_curves.png", dpi=150); plt.show()

## 3 · Evaluation

Predictions are turned back into **character spans** (exactly what the extension masks) and compared with the gold spans:

| Metric | Meaning |
| --- | --- |
| **Exact F1** | span boundaries *and* label must match exactly |
| **Partial F1** | label matches and the spans overlap |
| **Privacy recall** | share of gold values fully covered by *any* predicted span, i.e. nothing leaks, whatever the label |
| **Over-mask rate** | share of predicted spans that touch no gold value (unnecessary masking) |
| **Keep respected** | share of public entities (Google, India…) left unmasked |

In [ ]:
WORD = set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789")

def logits_torch(model):
    model.eval().to(DEVICE)
    def run(batch):
        with torch.no_grad():
            return model(**{k: v.to(DEVICE) for k, v in batch.items()}).logits.float().cpu().numpy()
    return run

def predict_spans(texts, run, batch_size=32, max_len=None):
    max_len = max_len or MAX_LEN
    results = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i + batch_size]
        enc = tokenizer(chunk, truncation=True, max_length=max_len, stride=32, padding=True,
                        return_overflowing_tokens=True, return_offsets_mapping=True, return_tensors="pt")
        mapping, offsets = enc.pop("overflow_to_sample_mapping").tolist(), enc.pop("offset_mapping").tolist()
        preds = run(enc).argmax(-1)
        per_text = [[] for _ in chunk]
        for w, sample in enumerate(mapping):
            cur = None
            for (s, e), p, m in zip(offsets[w], preds[w], enc["attention_mask"][w].tolist()):
                if not m or e <= s:
                    continue
                lab = id2label[int(p)]
                if lab == "O":
                    cur = None; continue
                tag, typ = lab.split("-", 1)
                if cur and cur["label"] == typ and (tag == "I" or s <= cur["end"]):
                    cur["end"] = e
                else:
                    cur = {"start": s, "end": e, "label": typ}; per_text[sample].append(cur)
        for text, spans in zip(chunk, per_text):
            merged = []
            for sp in sorted(spans, key=lambda x: (x["start"], -x["end"])):
                while sp["start"] > 0 and text[sp["start"] - 1] in WORD: sp["start"] -= 1   # widen to whole words
                while sp["end"] < len(text) and text[sp["end"]] in WORD: sp["end"] += 1
                if merged and sp["start"] <= merged[-1]["end"] and sp["label"] == merged[-1]["label"]:
                    merged[-1]["end"] = max(merged[-1]["end"], sp["end"])
                elif not merged or sp["start"] >= merged[-1]["end"]:
                    merged.append(dict(sp))
            results.append(merged)
    return results

def overlap(a, b):
    return a["start"] < b["end"] and b["start"] < a["end"]

def score(records, preds):
    exact = collections.Counter(); part = collections.Counter(); per_label = collections.defaultdict(collections.Counter)
    covered = gold_n = keep_ok = keep_n = over = pred_n = 0
    for rec, pr in zip(records, preds):
        gold = rec["spans"]
        for g in gold:
            gold_n += 1
            mask = np.zeros(g["end"] - g["start"], bool)
            for p in pr:
                lo, hi = max(p["start"], g["start"]), min(p["end"], g["end"])
                if lo < hi: mask[lo - g["start"]:hi - g["start"]] = True
            nonspace = np.array([not c.isspace() for c in rec["text"][g["start"]:g["end"]]])
            covered += bool(mask[nonspace].all())
        for p in pr:
            pred_n += 1
            over += not any(overlap(p, g) for g in gold)
        for k in rec["keep"]:
            keep_n += 1; keep_ok += not any(overlap(k, p) for p in pr)
        for kind, match in (("exact", lambda p, g: p["start"] == g["start"] and p["end"] == g["end"]), ("partial", overlap)):
            c = exact if kind == "exact" else part
            used = set()
            for p in pr:
                hit = next((j for j, g in enumerate(gold) if j not in used and g["label"] == p["label"] and match(p, g)), None)
                if hit is None:
                    c["fp"] += 1
                    if kind == "partial": per_label[p["label"]]["fp"] += 1
                else:
                    used.add(hit); c["tp"] += 1
                    if kind == "partial": per_label[p["label"]]["tp"] += 1
            c["fn"] += len(gold) - len(used)
            if kind == "partial":
                for j, g in enumerate(gold):
                    if j not in used: per_label[g["label"]]["fn"] += 1
    f1 = lambda c: 2 * c["tp"] / max(1, 2 * c["tp"] + c["fp"] + c["fn"])
    summary = {"exact_f1": f1(exact), "partial_f1": f1(part), "privacy_recall": covered / max(1, gold_n),
               "over_mask_rate": over / max(1, pred_n), "keep_respected": keep_ok / keep_n if keep_n else float("nan"),
               "gold_spans": gold_n}
    labels = {lab: {"precision": c["tp"] / max(1, c["tp"] + c["fp"]), "recall": c["tp"] / max(1, c["tp"] + c["fn"]),
                    "f1": f1(c), "support": c["tp"] + c["fn"]} for lab, c in per_label.items()}
    return summary, labels

TESTS = ["test_synthetic", "test_ai4privacy", "test_realistic"]
run_ft = logits_torch(trainer.model)
results, label_results, preds_ft = {}, {}, {}
for name in TESTS:
    t0 = time.time()
    preds_ft[name] = predict_spans([r["text"] for r in data[name]], run_ft)
    results[(RUN_NAME, name)], label_results[name] = score(data[name], preds_ft[name])
    results[(RUN_NAME, name)]["ms_per_example"] = 1000 * (time.time() - t0) / len(data[name])
pd.DataFrame(results).T.round(3)

In [ ]:
# Baseline: the off-the-shelf CoNLL-2003 NER model the extension used before (names, places, organizations only).
if COMPARE_BASELINE:
    base = pipeline("token-classification", model="dslim/bert-base-NER", aggregation_strategy="simple",
                    stride=32, device=0 if DEVICE == "cuda" else -1)
    MAP = {"PER": "PERSON", "LOC": "LOCATION", "ORG": "ORGANIZATION"}
    for name in TESTS:
        out = base([r["text"] for r in data[name]], batch_size=32)
        preds = [[{"start": e["start"], "end": e["end"], "label": MAP[e["entity_group"]]}
                  for e in ents if e["entity_group"] in MAP] for ents in out]
        results[("baseline: bert-base-NER (CoNLL)", name)], _ = score(data[name], preds)
    del base
table = pd.DataFrame(results).T
table.round(3)

In [ ]:
# Per-label results (partial match) on the held-out synthetic templates and the realistic set
rows = []
for name in ["test_synthetic", "test_realistic", "test_ai4privacy"]:
    for lab, m in sorted(label_results[name].items()):
        rows.append({"test set": name, "label": lab, **{k: round(v, 3) if isinstance(v, float) else v for k, v in m.items()}})
per_label = pd.DataFrame(rows)
per_label.pivot(index="label", columns="test set", values="f1").round(3)

In [ ]:
# Confusion matrix at entity level: what each gold value was predicted as (MISSED = no overlapping prediction)
pairs = []
for name in ["test_synthetic", "test_realistic"]:
    for rec, pr in zip(data[name], preds_ft[name]):
        for g in rec["spans"]:
            best = max(pr, key=lambda p: min(p["end"], g["end"]) - max(p["start"], g["start"]), default=None)
            hit = best and overlap(best, g)
            pairs.append((g["label"], best["label"] if hit else "MISSED"))
        for p in pr:
            if not any(overlap(p, g) for g in rec["spans"]):
                pairs.append(("NOT PII", p["label"]))
rows_l = LABELS + ["NOT PII"]; cols_l = LABELS + ["MISSED"]
cm = pd.DataFrame(0, index=rows_l, columns=cols_l)
for g, p in pairs:
    cm.loc[g, p] += 1
fig, ax = plt.subplots(figsize=(10, 8))
norm = cm.div(cm.sum(1).replace(0, 1), axis=0)
ax.imshow(norm.values, cmap="Greens", vmin=0, vmax=1)
ax.set_xticks(range(len(cols_l)), cols_l, rotation=60, ha="right"); ax.set_yticks(range(len(rows_l)), rows_l)
for i in range(len(rows_l)):
    for j in range(len(cols_l)):
        if cm.iloc[i, j]:
            ax.text(j, i, cm.iloc[i, j], ha="center", va="center", fontsize=7, color="white" if norm.iloc[i, j] > 0.6 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("gold"); ax.set_title("Entity-level confusion (synthetic + realistic tests)")
plt.tight_layout(); plt.savefig(OUT / "confusion_matrix.png", dpi=150); plt.show()

In [ ]:
# Error analysis on the realistic set: model output vs. gold for every example that is not perfect
def show(text, spans):
    out, pos = "", 0
    for s in sorted(spans, key=lambda x: x["start"]):
        out += text[pos:s["start"]] + f"[{text[s['start']:s['end']]}|{s['label']}]"; pos = s["end"]
    return (out + text[pos:]).replace("\n", " ⏎ ")

errors = []
for rec, pr in zip(data["test_realistic"], preds_ft["test_realistic"]):
    gold = {(g["start"], g["end"], g["label"]) for g in rec["spans"]}
    if gold != {(p["start"], p["end"], p["label"]) for p in pr}:
        errors.append(f"GOLD: {show(rec['text'], rec['spans'])}\nPRED: {show(rec['text'], pr)}\n")
print(f"{len(errors)} of {len(data['test_realistic'])} realistic examples differ from gold\n")
print("\n".join(errors[:15]))
(OUT / "realistic_errors.txt").write_text("\n".join(errors), encoding="utf-8")

## 4 · Export to ONNX and quantize to int8 (for the browser)

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType
import onnxruntime as ort

export = OUT / "hub"
(export / "onnx").mkdir(parents=True, exist_ok=True)
cpu_model = AutoModelForTokenClassification.from_pretrained(OUT / "model").eval()
sample = tokenizer(["Hi Priya, call me on +91 98450 12345 or mail priya@example.org"], return_tensors="pt")
names = [n for n in ("input_ids", "attention_mask", "token_type_ids") if n in sample]

class Logits(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, *xs): return self.m(**dict(zip(names, xs))).logits

kw = dict(input_names=names, output_names=["logits"], opset_version=14, do_constant_folding=True,
          dynamic_axes={**{n: {0: "batch", 1: "sequence"} for n in names}, "logits": {0: "batch", 1: "sequence"}})
if "dynamo" in inspect.signature(torch.onnx.export).parameters:
    kw["dynamo"] = False
# The exporter restores the wrapper's original train/eval mode afterwards, so the wrapper must be in eval mode,
# otherwise dropout stays switched on in the PyTorch model used for comparison below.
torch.onnx.export(Logits(cpu_model).eval(), tuple(sample[n] for n in names), str(export / "onnx" / "model.onnx"), **kw)
cpu_model.eval()
quantize_dynamic(str(export / "onnx" / "model.onnx"), str(export / "onnx" / "model_quantized.onnx"), weight_type=QuantType.QUInt8)

def logits_onnx(path):
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    return lambda batch: sess.run(["logits"], {n: batch[n].numpy() for n in names})[0]

with torch.no_grad():
    ref = cpu_model(**sample).logits.numpy()
for f in ["model.onnx", "model_quantized.onnx"]:
    got = logits_onnx(export / "onnx" / f)({n: sample[n] for n in names})
    print(f"{f:22} {os.path.getsize(export / 'onnx' / f) / 1e6:6.1f} MB   max |Δlogit| vs PyTorch = {np.abs(got - ref).max():.4f}   "
          f"same argmax = {(got.argmax(-1) == ref.argmax(-1)).mean():.1%}")

In [ ]:
# Accuracy and CPU latency: PyTorch fp32 vs ONNX fp32 vs ONNX int8 (what the browser runs)
DEVICE_SAVED, DEVICE = DEVICE, "cpu"          # latency on CPU, like the browser
variants = {"pytorch fp32": logits_torch(cpu_model), "onnx fp32": logits_onnx(export / "onnx" / "model.onnx"),
            "onnx int8": logits_onnx(export / "onnx" / "model_quantized.onnx")}
quant = {}
for vname, run in variants.items():
    for name in ["test_synthetic", "test_realistic"]:
        texts = [r["text"] for r in data[name]]
        t0 = time.time(); pr = predict_spans(texts, run, batch_size=1)
        s, _ = score(data[name], pr); s["cpu_ms_per_example"] = 1000 * (time.time() - t0) / len(texts)
        quant[(vname, name)] = s
DEVICE = DEVICE_SAVED
pd.DataFrame(quant).T[["exact_f1", "partial_f1", "privacy_recall", "keep_respected", "cpu_ms_per_example"]].round(3)

## 5 · Model card, results and upload

In [ ]:
def md_table(df):
    df = df.round(3).reset_index()
    return "\n".join(["| " + " | ".join(map(str, df.columns)) + " |", "|" + " --- |" * len(df.columns)] +
                     ["| " + " | ".join(map(str, r)) + " |" for r in df.values])

summary = table.copy(); summary.index = [f"{m} · {t}" for m, t in summary.index]
card = f'''---
license: apache-2.0
language: [en]
pipeline_tag: token-classification
library_name: transformers.js
tags: [pii, privacy, ner, onnx, india]
datasets: [{HF_DATASET}]
base_model: {BASE_MODEL}
---

# {RUN_NAME}

`{BASE_MODEL}` fine-tuned to detect personal data (PII) in web-page text for the
[Mintx Assistant](https://github.com/preetz-6/Mintx-) browser extension. It runs **in the browser** with
transformers.js (`onnx/model_quantized.onnx`, int8), so the text it reads never leaves the device.

**Labels (BIO):** {", ".join(LABELS)}. Public entities such as countries, large organizations and public figures are
deliberately left unlabelled (see the dataset's labelling policy).

## Results

{md_table(summary[["exact_f1", "partial_f1", "privacy_recall", "over_mask_rate", "keep_respected"]])}

*Privacy recall* = share of personal values fully covered by a predicted span. *Keep respected* = share of public
entities left visible. `test_realistic` is small (hand-checked, {len(data["test_realistic"])} examples).

## Training

{len(data["train"]):,} examples ({len(train_ds):,} after oversampling rare labels ×{RARE_OVERSAMPLE}), {EPOCHS} epochs,
lr {LR}, batch {BATCH}, max length {MAX_LEN}, {train_minutes:.0f} min on {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}.

## Data and attribution

Trained on [{HF_DATASET}](https://huggingface.co/datasets/{HF_DATASET}), which includes data derived from
*OpenPII 1.5M* by Ai4Privacy / Ai Suisse SA (CC-BY-4.0). All personal data in training is synthetic.

## Limitations

English only; Indian and Western name and number formats. Not a guarantee: use alongside rule-based detectors.
'''
for f in ["config.json", "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json", "vocab.txt",
          "spm.model", "added_tokens.json", "model.safetensors"]:
    if (OUT / "model" / f).exists():
        shutil.copy(OUT / "model" / f, export / f)
(export / "README.md").write_text(card, encoding="utf-8")

report = {"run": RUN_NAME, "base_model": BASE_MODEL, "epochs": EPOCHS, "train_minutes": round(train_minutes, 1),
          "results": {f"{m} | {t}": v for (m, t), v in results.items()},
          "quantization": {f"{m} | {t}": v for (m, t), v in quant.items()},
          "per_label": per_label.to_dict(orient="records")}
(OUT / "results.json").write_text(json.dumps(report, indent=2, default=float), encoding="utf-8")
print(json.dumps(report["results"], indent=1, default=float)[:3000])

In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import HfApi
    token = hf_token()
    assert token, "Add an HF_TOKEN secret (a Hugging Face *write* token) to push the model."
    api = HfApi(token=token)
    api.create_repo(HF_MODEL_REPO, exist_ok=True)
    api.upload_folder(folder_path=str(export), repo_id=HF_MODEL_REPO, commit_message=f"{RUN_NAME}: fine-tuned PII model")
    print(f"✅ https://huggingface.co/{HF_MODEL_REPO}")

import zipfile
archive = WORK / f"{RUN_NAME}_results.zip"
with zipfile.ZipFile(archive, "w") as z:
    for f in ["results.json", "training_curves.png", "confusion_matrix.png", "realistic_errors.txt"]:
        z.write(OUT / f, f)
print("📦 download this and send me results.json:", archive)